# KAGAMI D-160B — ekstraksi prakiraan operasional NOAA CFSv2 (prate, tmax) → agregat provinsi mingguan
Persetujuan Danen 2026-11-26. Sumber: NCEI THREDDS fileServer (byte-range 56 hari pertama).

In [ ]:
!pip -q install eccodes requests


In [ ]:
MAPPING_JSON = '{"lat_expected_first":89.27671287810583,"lon_step":0.9375,"mapping":[[1,89,102],[1,89,103],[1,89,104],[1,90,103],[1,90,104],[1,91,104],[2,91,105],[2,91,106],[2,92,105],[2,92,106],[2,93,104],[2,93,106],[2,94,106],[2,95,105],[3,95,107],[3,96,108],[4,92,107],[4,93,107],[4,93,108],[4,94,107],[4,94,108],[4,94,109],[4,95,108],[4,95,109],[4,95,110],[5,94,111],[6,96,109],[6,96,110],[6,96,111],[6,97,109],[7,97,108],[7,98,109],[7,99,110],[8,97,110],[8,97,111],[8,97,112],[8,98,110],[8,98,111],[8,98,112],[8,98,113],[8,99,111],[9,97,113],[10,99,112],[10,100,111],[10,100,112],[11,101,113],[12,102,114],[12,102,115],[12,102,116],[13,101,114],[14,102,117],[14,102,118],[14,102,119],[15,103,118],[16,102,120],[16,102,121],[16,103,119],[16,103,120],[16,103,121],[16,103,122],[17,104,123],[18,104,125],[19,105,128],[19,105,132],[20,93,117],[20,93,120],[20,94,117],[20,94,118],[20,94,119],[20,94,120],[20,94,121],[20,95,117],[20,95,118],[20,95,119],[20,95,120],[20,96,118],[20,97,118],[21,97,123],[21,97,124],[21,98,123],[21,98,124],[22,94,122],[22,95,121],[22,95,122],[22,96,119],[22,96,120],[22,96,121],[22,96,122],[22,96,123],[22,97,119],[22,97,120],[22,97,121],[22,97,122],[22,98,120],[22,98,122],[23,92,125],[23,93,122],[23,93,124],[23,93,125],[23,93,126],[23,94,123],[23,94,124],[23,94,125],[23,95,123],[23,95,124],[23,95,125],[23,96,124],[24,90,124],[24,90,125],[24,91,124],[24,91,125],[24,92,123],[24,92,124],[24,93,123],[25,94,130],[26,97,128],[26,97,129],[26,98,128],[26,99,128],[26,100,128],[27,98,130],[27,99,130],[27,99,131],[27,100,130],[27,100,131],[28,94,128],[28,94,129],[28,95,128],[28,95,130],[28,96,128],[28,96,129],[28,96,130],[28,96,131],[29,94,132],[30,98,127],[31,98,135],[31,98,137],[31,98,138],[31,98,139],[32,92,137],[32,93,137],[32,94,137],[32,95,136],[33,97,146],[33,97,147],[33,97,148],[33,97,149],[33,98,145],[33,98,146],[33,98,147],[33,98,148],[33,98,149],[33,98,150],[33,99,144],[33,99,145],[33,99,146],[33,99,147],[33,99,148],[33,99,149],[33,99,150],[33,100,147],[33,100,148],[33,100,149],[33,100,150],[33,101,148],[33,101,149],[33,101,150],[33,102,148],[33,102,149],[33,102,150],[33,103,147],[33,103,148],[33,103,149],[33,103,150],[34,95,141],[34,96,140],[34,96,141],[34,96,142],[34,96,143],[34,97,143],[34,98,142],[34,98,143]]}'

import os, json, struct, time, hashlib, datetime as dt
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import eccodes
# Run 2026-11-26: drive.mount gagal (timeout menunggu popup OAuth) -> output lokal /content, hasil diambil lewat files.download (sel terakhir).
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/KAGAMI' if os.path.isdir('/content/drive/MyDrive/KAGAMI') else '/content/drive/MyDrive'
else:
    ROOT = '/content'
OUT = os.path.join(ROOT, 'experiments', 'd160_cfsv2'); PARTS = os.path.join(OUT, 'parts')
os.makedirs(PARTS, exist_ok=True); print('OUT', OUT)

MAP = json.loads(MAPPING_JSON)
LAT_EXPECTED_FIRST = MAP['lat_expected_first']
CELLS = np.array(MAP['mapping'])               # [province_id, j, i]
PIDS = sorted(set(CELLS[:, 0].tolist()))
BASE = 'https://www.ncei.noaa.gov/thredds/fileServer/model-cfs_v2_for_ts'
DAYS, STEP_H = 56, 6
NEED_H = DAYS * 24
S = requests.Session(); S.headers.update({'User-Agent': 'kagami-research-client'})

def url(day, var):
    s = day.strftime('%Y%m%d')
    return f"{BASE}/{s[:4]}/{s[:6]}/{s}/{s}00/{var}.01.{s}00.daily.grb2"

def fhour(msg):
    L1 = struct.unpack('>I', msg[16:20])[0]; p = 16 + L1
    while p < len(msg) - 4:
        sl = struct.unpack('>I', msg[p:p+4])[0]; sn = msg[p+4]
        if sn == 4:
            unit = msg[p+17]; v = struct.unpack('>I', msg[p+18:p+22])[0]
            return v * {0: 1/60, 1: 1, 2: 24}.get(unit, 1)
        p += sl
    return None

def fetch_msgs(u):
    buf, start, chunk = b'', 0, 9_500_000
    msgs, off = [], 0
    while True:
        r = S.get(u, headers={'Range': f'bytes={start}-{start + chunk - 1}'}, timeout=300)
        if r.status_code not in (200, 206):
            raise RuntimeError(f'HTTP {r.status_code}')
        buf += r.content; start += len(r.content)
        while off + 16 <= len(buf):
            L = struct.unpack('>Q', buf[off+8:off+16])[0]
            if off + L > len(buf): break
            m = buf[off:off+L]; h = fhour(m)
            msgs.append((h, m)); off += L
            if h is not None and h >= NEED_H:
                return msgs, buf[:off]
        if len(r.content) < chunk:
            return msgs, buf[:off]
        chunk = 3_000_000

def decode(msgs, kind):
    daily = np.full((DAYS, 190, 384), np.nan, np.float32); cnt = np.zeros(DAYS)
    for h, m in msgs:
        if h is None or h <= 0 or h > NEED_H: continue
        d = int((h - 1) // 24)
        gid = eccodes.codes_new_from_message(bytes(m))
        try:
            v = eccodes.codes_get_values(gid).reshape(190, 384).astype(np.float32)
        finally:
            eccodes.codes_release(gid)
        if kind == 'prate':
            daily[d] = np.nan_to_num(daily[d]) + v * 86400 / 4      # mm/hari: rerata 4 laju 6 jam
        else:
            daily[d] = v - 273.15 if cnt[d] == 0 else np.fmax(daily[d], v - 273.15)
        cnt[d] += 1
    return daily, cnt

def province_means(daily):
    out = np.full((len(PIDS), DAYS), np.nan)
    for k, pid in enumerate(PIDS):
        c = CELLS[CELLS[:, 0] == pid]
        out[k] = np.nanmean(daily[:, c[:, 1], c[:, 2]], axis=1)
    return out

def check_grid(m):
    gid = eccodes.codes_new_from_message(bytes(m))
    try:
        lat = eccodes.codes_get_array(gid, 'latitudes').reshape(190, 384)[:, 0]
    finally:
        eccodes.codes_release(gid)
    assert abs(lat[0] - LAT_EXPECTED_FIRST) < 0.01, lat[:3]
    return float(lat[0])

def one_week(week):
    part = os.path.join(PARTS, f"{week.date()}.json")
    if os.path.exists(part):
        return 'skip'
    rec = {'week': str(week.date())}
    for off_days in (4, 3, 2):                              # Jumat, Kamis, Rabu 00Z
        day = week + pd.Timedelta(days=off_days)
        try:
            res = {}
            for var in ('prate', 'tmax'):
                msgs, raw = fetch_msgs(url(day, var))
                res[var] = (msgs, raw)
            rec['init'] = day.strftime('%Y-%m-%dT00')
            break
        except Exception as e:
            rec[f'err_{off_days}'] = str(e)[:120]
    else:
        json.dump(rec, open(part, 'w')); return 'missing'
    P, cp = decode(res['prate'][0], 'prate'); T, ct = decode(res['tmax'][0], 'tmax')
    pp, pt = province_means(P), province_means(T)
    rows = []
    for k, pid in enumerate(PIDS):
        rows.append({'province_id': int(pid), 'r_1_28': float(np.nansum(pp[k, :28])), 'r_29_56': float(np.nansum(pp[k, 28:])),
                     'r_1_56': float(np.nansum(pp[k])), 'rmax_1_28': float(np.nanmax(pp[k, :28])),
                     'tx_1_28': float(np.nanmean(pt[k, :28])), 'tx_29_56': float(np.nanmean(pt[k, 28:]))})
    rec.update({'rows': rows, 'days_prate_complete': int((cp == 4).sum()), 'days_tmax_complete': int((ct == 4).sum()),
                'bytes_prate': len(res['prate'][1]), 'bytes_tmax': len(res['tmax'][1]),
                'sha_prate': hashlib.sha256(res['prate'][1]).hexdigest(), 'sha_tmax': hashlib.sha256(res['tmax'][1]).hexdigest()})
    np.savez_compressed(os.path.join(PARTS, f"{week.date()}_daily.npz"), prate=pp.astype(np.float32), tmax=pt.astype(np.float32))
    json.dump(rec, open(part, 'w'))
    return 'ok'

# uji satu run + cek grid
m0, _ = fetch_msgs(url(pd.Timestamp('2022-12-23'), 'prate'))
print('cek grid lat0', check_grid(m0[0][1]), 'pesan', len(m0), 'jam terakhir', m0[-1][0])

weeks = list(pd.date_range('2017-12-25', '2026-11-16', freq='W-MON'))
t0 = time.time(); done = {}
with ThreadPoolExecutor(6) as ex:
    futs = {ex.submit(one_week, w): w for w in weeks}
    for i, f in enumerate(as_completed(futs), 1):
        try:
            s = f.result()
        except Exception as e:
            s = 'error: ' + str(e)[:100]
        done[s.split(':')[0]] = done.get(s.split(':')[0], 0) + 1
        if i % 25 == 0:
            print(i, len(weeks), round(time.time() - t0), 's', done, flush=True)
print('selesai', round(time.time() - t0), 's', done)


In [ ]:

import glob
recs = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(PARTS, '*.json')))]
rows = []
for r in recs:
    for x in r.get('rows', []):
        rows.append({'week': r['week'], 'init': r.get('init'), **x, 'days_prate_complete': r.get('days_prate_complete'),
                     'days_tmax_complete': r.get('days_tmax_complete')})
df = pd.DataFrame(rows)
man = pd.DataFrame([{k: r.get(k) for k in ('week', 'init', 'bytes_prate', 'bytes_tmax', 'sha_prate', 'sha_tmax', 'days_prate_complete', 'days_tmax_complete')} | {'errors': '; '.join(v for k, v in r.items() if k.startswith('err_'))} for r in recs])
df.to_csv(os.path.join(OUT, 'cfsv2_province_weekly.csv'), index=False)
man.to_csv(os.path.join(OUT, 'cfsv2_manifest.csv'), index=False)
print(df.shape, man['init'].isna().sum(), 'minggu tanpa run'); print(df.describe().round(2))
import shutil
shutil.make_archive(os.path.join(OUT, 'cfsv2_parts'), 'zip', PARTS)   # agregat harian per provinsi (npz) + json per run
from google.colab import files
for fn in ('cfsv2_province_weekly.csv', 'cfsv2_manifest.csv', 'cfsv2_parts.zip'):
    files.download(os.path.join(OUT, fn))
